In [1]:
import os
from pathlib import Path
import sys
import pandas as pd

_root_override = os.environ.get("THESIS_REPO_ROOT")
if _root_override:
    _bootstrap_root = Path(_root_override).expanduser().resolve()
else:
    _bootstrap_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / ".git").exists()), None)
if _bootstrap_root is None:
    raise RuntimeError("Repository root not found; set THESIS_REPO_ROOT or start Jupyter inside the repository.")
if str(_bootstrap_root) not in sys.path:
    sys.path.insert(0, str(_bootstrap_root))

from src.thesis_pipeline.paths import (
    DATA_RAW_DIR, METADATA_DIR, OUTPUTS_DIR, PHASE3_DIR, REPO_ROOT,
    SELECTED_RAW_DIR,
)

PROJECT_DIR = REPO_ROOT
DATA_DIR = DATA_RAW_DIR
DATA_SEL_DIR = SELECTED_RAW_DIR
META_DIR = METADATA_DIR
OUT_DIR = OUTPUTS_DIR

WEGE_PATH = DATA_RAW_DIR / "MiD2023_Wege.csv"
WEGE_SEL_PATH = SELECTED_RAW_DIR / "trips_selected_raw.csv"
AUTOS_PATH = DATA_RAW_DIR / "MiD2023_Autos.csv"

MNL_WIDE_PATH = PHASE3_DIR / "mnl_vehicle_choice_wide_base.csv"
PHASE3_FUNNEL_PATH = PHASE3_DIR / "phase3_sample_funnel.csv"
OCCASION_PATH = PHASE3_DIR / "vehicle_choice_occasions_classified.csv"

In [2]:
occasion_classification = pd.read_csv(
    OCCASION_PATH, 
    sep=",",
    encoding="latin-1",
    dtype=str
)
print(occasion_classification.head())
print(occasion_classification.columns.tolist())
print(len(occasion_classification["CHOICE_ID"]))
print(occasion_classification["AVAILABLE_CAR_SET"].value_counts())

             CHOICE_ID              TRIP_ID SOURCE_ROW_ID      H_ID     HP_ID  \
0  10000480_10000482_1  10000480_10000482_1            25  10000480  10000482   
1  10000600_10000601_5  10000600_10000601_5            47  10000600  10000601   
2  10000600_10000602_1  10000600_10000602_1            50  10000600  10000602   
3  10000600_10000601_3  10000600_10000601_3            45  10000600  10000601   
4  10001310_10001311_1  10001310_10001311_1            87  10001310  10001311   

  P_ID W_ID START_MIN ARRIVAL_MIN HOME_ORIGIN  ...  \
0    2    1       545         625           1  ...   
1    1    5      1005        1020           1  ...   
2    2    1       435         455           1  ...   
3    1    3       905         925           1  ...   
4    1    1       330         350           1  ...   

  EXCL_SIMULTANEOUS_DEPARTURE EXCL_HOUSEHOLD_TIME_UNRESOLVED  \
0                           0                              0   
1                           0                              0

In [3]:
phase3_funnel = pd.read_csv(
    PHASE3_FUNNEL_PATH,
    sep=",",
    encoding="latin-1",
    dtype=str
)
print(phase3_funnel.head())

  step_number                    step_name input_occasions  \
0           1             INCOMPLETE_FLEET           56340   
1           2            UNMATCHED_VEHICLE           44066   
2           3         DUPLICATE_IDENTIFIER           44066   
3           4  INCOMPLETE_CHOICE_TRIP_TIME           44066   
4           5             INVALID_SNAPSHOT           43812   

  newly_excluded_occasions remaining_occasions remaining_households  \
0                    12274               44066                26591   
1                        0               44066                26591   
2                        0               44066                26591   
3                      254               43812                26483   
4                        0               43812                26483   

  share_of_initial_occasions share_of_previous_step  
0         0.7821441249556266     0.7821441249556266  
1         0.7821441249556266                    1.0  
2         0.7821441249556266          

In [5]:
mnl_check = pd.read_csv(
    MNL_WIDE_PATH,
    sep=",",
    encoding="latin-1",
    dtype=str
)

print(mnl_check.columns.tolist())
print(mnl_check.head())
print(len(mnl_check["CHOICE_ID"]))

mnl_check.to_excel(OUT_DIR / "mnl_check.xlsx", index=False)

['CHOICE_ID', 'SOURCE_ROW_ID', 'H_ID', 'HP_ID', 'P_ID', 'W_ID', 'CHOICE', 'AV_1', 'AV_2', 'START_MIN', 'ARRIVAL_MIN', 'W_ZWECK', 'W_SO1', 'W_GEW']
             CHOICE_ID SOURCE_ROW_ID      H_ID     HP_ID P_ID W_ID CHOICE  \
0  10000480_10000482_1            25  10000480  10000482    2    1      2   
1  10000600_10000601_5            47  10000600  10000601    1    5      1   
2  10000600_10000601_3            45  10000600  10000601    1    3      1   
3  10001310_10001311_1            87  10001310  10001311    1    1      1   
4  10001400_10001403_1           148  10001400  10001403    3    1      2   

  AV_1 AV_2 START_MIN ARRIVAL_MIN W_ZWECK W_SO1               W_GEW  
0    1    1       545         625       7     1  0.1455903752720406  
1    1    1      1005        1020       6   809   0.721786426193377  
2    1    1       905         925      10   809   0.721786426193377  
3    1    1       330         350       1     1  2.8639436453562603  
4    1    1       450         495       

In [9]:
auto = pd.read_csv(
    AUTOS_PATH,
    sep=",",
    encoding="latin-1",
    dtype=str
)

print(auto["A_ID"].value_counts())

A_ID
1    189343
2     85408
3     16179
Name: count, dtype: int64


In [10]:
wege_sel = pd.read_csv(
    WEGE_SEL_PATH,
    sep=",",
    encoding="latin-1",
    dtype=str
)

print(len(wege_sel["W_ID"]))
print(wege_sel["W_SO1"].value_counts())
print(wege_sel["W_WAUTO"].value_counts())
print(wege_sel["W_ID"].value_counts())

1087393
W_SO1
809    679963
1      271993
701    107368
2       18254
9        5576
810      4239
Name: count, dtype: int64
W_WAUTO
202    389963
706    307709
1      210877
2       72933
701     65283
4       29943
3        7616
708      1437
9        1101
995       531
Name: count, dtype: int64
W_ID
1      300062
2      271787
3      168021
4      117229
5       58233
        ...  
146       285
147       240
148       183
149        83
150        63
Name: count, Length: 62, dtype: int64


In [14]:
wege = pd.read_csv(
    WEGE_PATH,
    sep=",",
    encoding="latin-1",
    dtype=str
)


print(wege["W_SO1"].value_counts())
print(len(wege["W_ID"]))

W_SO1
809    679963
1      271993
701    107368
2       18254
9        5576
810      4239
Name: count, dtype: int64
1087393


In [ ]:
check = (
    wege.assign(is_first=(wege["W_SO1"] == "1"))
        .groupby("HP_ID")["is_first"]
        .sum()
)

print(check.head())

problem = check[check > 1]

print(problem)
print("人数：", len(problem))
print(check.value_counts().sort_index())

HP_ID
10          1
10000011    1
10000191    0
10000192    0
10000371    1
Name: is_first, dtype: int64
Series([], Name: is_first, dtype: int64)
人数： 0
is_first
0     33213
1    271993
Name: count, dtype: int64
